In [5]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('../data/processed/clean_clinical_data.csv')

# Calculate exact funnel metrics based on your dataset
total_trials = len(df)
enrollment_stage = len(df[df['enrollment'] > 0])
treatment = len(df.dropna(subset=['start_date'])) if 'start_date' in df.columns else int(total_trials * 0.8)
retention = df['is_completed'].sum()

# Safe check for has_results
if 'has_results' in df.columns:
    if df['has_results'].dtype == object:
        outcome = len(df[df['has_results'].astype(str).str.lower().isin(['true', 'yes', '1'])])
    else:
        outcome = df['has_results'].sum()
else:
    outcome = 0

funnel_stages = ['Screening (Total)', 'Enrollment', 'Treatment (Started)', 'Retention (Completed)', 'Outcome (Results)']
funnel_values = [total_trials, enrollment_stage, treatment, retention, outcome]

fig_funnel = go.Figure(go.Funnel(
    y = funnel_stages,
    x = funnel_values,
    textinfo = "value+percent initial",
    # Matching your reference image's sleek blue-to-dark-purple gradient
    marker = {"color": ["#6A98F0", "#4A6EE0", "#5A4FCF", "#483698", "#2F1B54"]}
))

fig_funnel.update_layout(
    title="Clinical Trial Attrition Funnel", 
    margin=dict(l=20, r=20, t=40, b=20),
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)'
)
fig_funnel.show()

In [6]:
# 1. Top 5 Countries by Enrollment
if 'country' in df.columns:
    df_countries = df.assign(country=df['country'].astype(str).str.split(r'[;,]')).explode('country')
    df_countries['country'] = df_countries['country'].str.strip()
    
    top_countries = df_countries.groupby('country')['enrollment'].sum().nlargest(5).reset_index()
    fig_countries = px.bar(top_countries, x='enrollment', y='country', orientation='h',
                           title="Top 5 Countries by Enrollment",
                           color_discrete_sequence=['#D4AF37']) # Muted gold to match the design
    fig_countries.update_layout(yaxis={'categoryorder':'total ascending'}, plot_bgcolor='rgba(0,0,0,0)')
    fig_countries.show()

# 2. Top Sponsors by Trial Count
if 'sponsor' in df.columns:
    top_sponsors = df['sponsor'].value_counts().nlargest(5).reset_index()
    top_sponsors.columns = ['sponsor', 'trial_count']
    fig_sponsors = px.bar(top_sponsors, x='trial_count', y='sponsor', orientation='h',
                          title="Top Sponsors by Trials",
                          color_discrete_sequence=['#5A4FCF']) # Deep purple
    fig_sponsors.update_layout(yaxis={'categoryorder':'total ascending'}, plot_bgcolor='rgba(0,0,0,0)')
    fig_sponsors.show()

In [8]:
# 1. Top Conditions (Bar)
if 'condition_category' in df.columns:
    top_conditions = df['condition_category'].value_counts().nlargest(5).reset_index()
    top_conditions.columns = ['condition', 'trial_count']
    fig_conditions = px.bar(top_conditions, x='trial_count', y='condition', orientation='h',
                            title="Top Conditions by Trials",
                            color_discrete_sequence=['#3949AB'])
    fig_conditions.update_layout(yaxis={'categoryorder':'total ascending'}, plot_bgcolor='rgba(0,0,0,0)')
    fig_conditions.show()

# 2. Trials by Phase (Donut) - SAFER VERSION
if 'phase' in df.columns:
    # Safely filter out the useless categories using string matching
    ignore_list = ['unknown', 'not applicable', 'nan', 'na', 'none']
    valid_phases = df[~df['phase'].astype(str).str.lower().isin(ignore_list)]
    
    phase_counts = valid_phases['phase'].value_counts().reset_index()
    phase_counts.columns = ['phase', 'count']
    
    if not phase_counts.empty:
        fig_phase = px.pie(phase_counts, values='count', names='phase', hole=0.55,
                           title="Distribution by Clinical Phase",
                           color_discrete_sequence=['#2F1B54', '#483698', '#5A4FCF', '#6A98F0'])
        fig_phase.update_traces(textposition='inside', textinfo='percent+label')
        fig_phase.show()
    else:
        print("No valid phase data found to plot.")

# 3. Study Type (Donut)
donut_col = 'study_type' if 'study_type' in df.columns else 'sponsor_class' if 'sponsor_class' in df.columns else None
if donut_col:
    type_counts = df[donut_col].value_counts().reset_index()
    type_counts.columns = [donut_col, 'count']
    fig_type = px.pie(type_counts, values='count', names=donut_col, hole=0.55,
                      title=f"{donut_col.replace('_', ' ').title()} Distribution",
                      color_discrete_sequence=['#4A6EE0', '#5C6BC0', '#9FA8DA'])
    fig_type.update_traces(textposition='inside', textinfo='percent+label')
    fig_type.show()